# Análisis exploratorio de datos — MIMIC-IV-ED

Importación de dependencias para análisis tabular, generación de informes automáticos (Sweetviz), visualización y preprocesamiento.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv
from sklearn.impute import KNNImputer
import os
from pathlib import Path
from dotenv import load_dotenv

# Buscar .env subiendo desde el CWD (funciona independientemente de donde se lance Jupyter)
_cur = Path.cwd()
while not (_cur / ".env").exists() and _cur != _cur.parent:
    _cur = _cur.parent
load_dotenv(_cur / ".env", override=True)

DATA  = Path(os.getenv("MIMIC_IV_ED_PATH", ""))
CLEAN = DATA / "clean_csv"
CLEAN.mkdir(parents=True, exist_ok=True)

print(f"DATA  : {DATA}")
print(f"Existe: {DATA.exists()}")

DATA  : C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\data
Existe: True


# Análisis exploratorio de datos (EDA) sobre las tablas MIMIC-IV-ED

Se genera un informe automático con Sweetviz para cada tabla del módulo de urgencias. El objetivo es identificar distribuciones, valores faltantes, cardinalidades y posibles inconsistencias antes de aplicar ninguna transformación.

## Tabla `edstays`

Tabla maestra de episodios de urgencias. Contiene una fila por estancia (`stay_id`), con metadatos demográficos, fechas de entrada/salida y disposición final del paciente.

In [2]:
df_edstays = pd.read_csv(DATA / 'edstays.csv')
reporte = sv.analyze(df_edstays)
reporte.show_html(str(DATA.parent / 'analisis_mimic_ed.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_ed.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Pipeline de limpieza — `edstays`

Se aplican tres transformaciones en cadena: estandarización de cadenas categóricas, conversión de fechas y optimización de tipos. El resultado se persiste en `clean_csv/`.

In [3]:
def clean_categorical_strings(df):
    """Estandariza columnas categóricas a mayúsculas y elimina espacios."""
    print("--- Paso 1: Limpiando textos categóricos ---")
    cols_to_fix = ['gender', 'race', 'arrival_transport', 'disposition']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.upper()
    return df

In [4]:
def convert_datetimes(df):
    """Convierte las columnas de tiempo a objetos datetime de Pandas."""
    print("--- Paso 2: Transformando fechas (intime, outtime) ---")
    if 'intime' in df.columns:
        df['intime'] = pd.to_datetime(df['intime'])
    if 'outtime' in df.columns:
        df['outtime'] = pd.to_datetime(df['outtime'])
    return df

In [5]:
def optimize_edstays_types(df):
    """Ajusta los tipos de datos e imputa el flag -1 para no hospitalizados."""
    print("--- Paso 3: Optimizando IDs y categorías ---")
    
    # IDs de paciente y estancia como enteros
    if 'subject_id' in df.columns:
        df['subject_id'] = df['subject_id'].astype(int)
    if 'stay_id' in df.columns:
        df['stay_id'] = df['stay_id'].astype(int)
        
    # Flag -1 para estancias sin ingreso hospitalario (hadm_id nulo)
    if 'hadm_id' in df.columns:
        df['hadm_id'] = df['hadm_id'].fillna(-1).astype(int)
            
    # Conversión a categoría para optimizar memoria
    cat_cols = ['gender', 'race', 'arrival_transport', 'disposition']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')
            
    return df

In [6]:
def run_edstays_pipeline(df_input):
    """Orquesta la limpieza del dataset edstays."""
    df = df_input.copy()
    
    # Normalización de nombres de columna
    df.columns = df.columns.str.strip()
    
    df = (
        df.pipe(clean_categorical_strings)
          .pipe(convert_datetimes)
          .pipe(optimize_edstays_types)
    )
    
    # Eliminación de duplicados exactos
    df = df.drop_duplicates()
    
    print(f"Pipeline de edstays completado. Filas finales: {len(df)}")
    return df

In [7]:
df_edstays_limpio = run_edstays_pipeline(df_edstays)
# Guardar el resultado
output_dir = "clean_csv"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

df_edstays_limpio.to_csv(os.path.join(output_dir, "edstays_cleaned.csv"), index=False)

--- Paso 1: Limpiando textos categóricos ---
--- Paso 2: Transformando fechas (intime, outtime) ---
--- Paso 3: Optimizando IDs y categorías ---
Pipeline de edstays completado. Filas finales: 425087


## Tabla `diagnosis`

Contiene los diagnósticos ICD codificados para cada episodio. Puede incluir múltiples filas por `stay_id` ordenadas por `seq_num`.

In [8]:
df_diagnosis = pd.read_csv(DATA / 'diagnosis.csv')
reporte = sv.analyze(df_diagnosis)
reporte.show_html(str(DATA.parent / 'analisis_mimic_di.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_di.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Limpieza y normalización de `diagnosis`

Se normalizan códigos ICD, se unifican versiones 9 y 10 bajo un identificador común, y se añade una columna de capítulo clínico para facilitar la agrupación por sistema orgánico.

In [9]:
def clean_diagnosis_dataset(df):
    """Normaliza códigos ICD, crea identificador unificado y asigna capítulo clínico por sistema orgánico."""
    df_diag = df.copy()
    
    # Eliminación de puntos y espacios en el código ICD
    df_diag['icd_code'] = df_diag['icd_code'].astype(str).str.replace('.', '', regex=False).str.strip()
    
    # Código unificado con prefijo de versión para evitar colisiones entre ICD-9 e ICD-10
    df_diag['unified_code'] = df_diag['icd_version'].astype(str) + "-" + df_diag['icd_code']
    
    # Normalización de títulos diagnósticos
    df_diag['icd_title'] = df_diag['icd_title'].str.lower().str.replace(r'[^\w\s]', '', regex=True).str.strip()
    
    # Asignación de capítulo clínico según sistema orgánico (ICD-9 e ICD-10)
    def get_chapter(row):
        code = str(row['icd_code'])
        version = row['icd_version']
        
        if version == 10:
            # En ICD-10, la primera letra identifica el sistema
            return code[0].upper()
        else:
            # En ICD-9, se mapean rangos numéricos a letras equivalentes de ICD-10
            try:
                prefix = int(code[:3])
                if 390 <= prefix <= 459: return 'I'  # Circulatorio
                if 460 <= prefix <= 519: return 'J'  # Respiratorio
                if 520 <= prefix <= 579: return 'K'  # Digestivo
                if 580 <= prefix <= 629: return 'N'  # Genitourinario
                if 800 <= prefix <= 999: return 'S'  # Lesiones
                return f"9-{str(prefix)[0]}"
            except (ValueError, IndexError):
                return "Other"

    df_diag['icd_chapter'] = df_diag.apply(get_chapter, axis=1)
    
    # Indicador de diagnóstico principal (primer motivo de la visita)
    df_diag['is_primary'] = (df_diag['seq_num'] == 1).astype(int)
    
    # Optimización de tipos de datos
    df_diag['seq_num'] = df_diag['seq_num'].astype('int8')
    df_diag['icd_version'] = df_diag['icd_version'].astype('category')
    df_diag['icd_chapter'] = df_diag['icd_chapter'].astype('category')
    
    return df_diag

In [10]:
# Ejecucion de la limpieza y exportacion del resultado
diagnosis_cleaned = clean_diagnosis_dataset(df_diagnosis)
diagnosis_cleaned.to_csv(CLEAN / "diagnosis_cleaned.csv", index=False)

## Tabla `medrecon`

Registro de medicación previa a la llegada al servicio de urgencias. Incluye nombre del fármaco, código GSN y clasificación terapéutica ETC.

In [11]:
df_medrecon = pd.read_csv(DATA / 'medrecon.csv')
reporte = sv.analyze(df_medrecon)
reporte.show_html(str(DATA.parent / 'analisis_mimic_me.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_me.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


Las columnas `name`, `gsn` y `ndc` son representaciones equivalentes del mismo fármaco y presentan correlación próxima a 1. Se eliminan `name` y `ndc`: la primera aporta cadenas de texto sin beneficio adicional respecto a `gsn`, y la segunda contiene valores numéricos de magnitud muy elevada que podrían introducir escala espuria en los modelos. Se conserva `gsn` por ser ya numérico y no requerir transformación.

La columna `etc_rn` presenta baja variabilidad (valores mayoritariamente repetidos) y no aporta información discriminativa relevante para el modelo.

La columna `etcdescription` contiene texto descriptivo libre que no aporta señal estructurada utilizable por los modelos de ML aplicados en este proyecto.

In [12]:
def clean_strings(df):
    """Estandariza nombres de medicamentos y descripciones a minúsculas."""
    print("--- Paso 1: Limpiando cadenas de texto ---")
    cols_to_fix = ['name', 'etcdescription']
    for col in cols_to_fix:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
    return df

In [13]:
def handle_missing_values(df):
    """Elimina registros sin clasificación terapéutica (etccode/etcdescription nulos)."""
    print("--- Paso 2: Tratando valores nulos ---")
    # etccode y etcdescription presentan 11.728 nulos; se eliminan esas filas
    df = df.dropna(subset=['etccode', 'etcdescription'])
    return df

In [14]:
def normalize_etc_mapping(df):
    """Resuelve la discrepancia entre etccode (1201 valores) y etcdescription (1205)."""
    print("--- Paso 3: Normalizando mapeo de códigos ETC ---")
    # Se asigna la descripción más frecuente para cada código único
    mapping = df.groupby('etccode')['etcdescription'].agg(lambda x: x.mode()[0]).to_dict()
    df['etcdescription'] = df['etccode'].map(mapping)
    return df

In [15]:
def optimize_types(df):
    """Optimiza el uso de memoria para el dataset de ~3 millones de filas."""
    print("--- Paso 4: Optimizando tipos de datos ---")
    # etc_rn tiene solo 5 valores distintos; se convierte a categoría
    if 'etc_rn' in df.columns:
        df['etc_rn'] = df['etc_rn'].astype('category')
    
    df['subject_id'] = df['subject_id'].astype(int)
    df['stay_id'] = df['stay_id'].astype(int)
    return df

In [16]:
def run_cleaning_pipeline(df_input):
    """Ejecuta el pipeline de limpieza de medrecon de forma secuencial."""
    df = df_input.copy()
    
    # Normalización de nombres de columna
    df.columns = df.columns.str.strip()
    
    df = (
        df.pipe(clean_strings)
          .pipe(handle_missing_values)
          .pipe(normalize_etc_mapping)
          .pipe(optimize_types)
    )
    
    df = df.drop_duplicates()
    print(f"\nProceso completado. Filas finales: {len(df)}")
    return df

In [17]:
df_medrecon_limpio = run_cleaning_pipeline(df_input=df_medrecon)

directory = "clean_csv"
if not os.path.exists(directory):
    os.makedirs(directory)

file_path = os.path.join(directory, "medrecon_cleaned.csv")
df_medrecon_limpio.to_csv(file_path, index=False)

print(f"Archivo guardado en: {file_path}")

--- Paso 1: Limpiando cadenas de texto ---
--- Paso 2: Tratando valores nulos ---
--- Paso 3: Normalizando mapeo de códigos ETC ---
--- Paso 4: Optimizando tipos de datos ---

Proceso completado. Filas finales: 2975603
Archivo guardado en: clean_csv\medrecon_cleaned.csv


## Tabla `pyxis`

Registro de dispensaciones realizadas desde el sistema Pyxis durante la estancia en urgencias. Permite identificar administración de vasopresores y otros fármacos de uso crítico.

In [18]:
df_pyxis = pd.read_csv(DATA / 'pyxis.csv')
reporte = sv.analyze(df_pyxis)
reporte.show_html(str(DATA.parent / 'analisis_mimic_py.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_py.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Limpieza de `pyxis`

Se extrae el nombre base del medicamento, se tratan los valores faltantes de GSN y se optimizan los tipos de datos.

In [19]:
def clean_pyxis_names(df):
    """Normaliza nombres de medicamentos a mayúsculas y extrae el nombre base (primera palabra)."""
    df_pyxis_clean = df.copy()
    
    # Normalización a mayúsculas y eliminación de espacios
    df_pyxis_clean['name'] = df_pyxis_clean['name'].astype(str).str.upper().str.strip()
    
    # Extracción del nombre base del medicamento (primera palabra)
    df_pyxis_clean['med_base_name'] = df_pyxis_clean['name'].str.split().str[0]
    
    # Eliminación de texto entre paréntesis y caracteres especiales al final
    df_pyxis_clean['med_base_name'] = df_pyxis_clean['med_base_name'].str.split('(').str[0]
    df_pyxis_clean['med_base_name'] = df_pyxis_clean['med_base_name'].str.replace(r'[^A-Z0-9]$', '', regex=True)
    
    return df_pyxis_clean

### Tratamiento de valores faltantes en GSN

Los valores no numéricos se convierten a `NaN`; los GSN ausentes reciben el valor centinela -1 para distinguirlos de códigos reales sin confundirlos con IDs válidos.

In [20]:
def clean_and_impute_gsn(df):
    """
    Convierte valores no numéricos a NaN e imputa con valor centinela -1.
    """
    df_pyxis_clean = df.copy()

    # Conversión de cadenas no numéricas a NaN
    df_pyxis_clean['gsn'] = pd.to_numeric(df_pyxis_clean['gsn'], errors='coerce')
    
    # Los valores 0 se tratan como ausentes según documentación
    df_pyxis_clean['gsn'] = df_pyxis_clean['gsn'].replace(0, np.nan)

    # Valor centinela -1 para GSN desconocido, distinguible de IDs reales
    df_pyxis_clean['gsn'] = df_pyxis_clean['gsn'].fillna(-1).astype('Int64')
    
    return df_pyxis_clean

### Eliminación de duplicados y optimización de tipos en `pyxis`

Se eliminan filas exactamente duplicadas y se reducen los tipos enteros para minimizar el uso de memoria en un dataset de ~1.6M filas.

In [21]:
def optimize_pyxis_dataset(df):
    """Elimina duplicados exactos y convierte tipos de datos para reducir uso de memoria."""
    df_optimized = df.drop_duplicates().copy()
    
    # Conversión de tipos para reducir uso de memoria
    df_optimized['charttime'] = pd.to_datetime(df_optimized['charttime'])
    df_optimized['subject_id'] = df_optimized['subject_id'].astype('int32')
    df_optimized['stay_id'] = df_optimized['stay_id'].astype('int32')
    df_optimized['med_rn'] = df_optimized['med_rn'].astype('int16')
    df_optimized['gsn_rn'] = df_optimized['gsn_rn'].astype('int16')
    
    # med_base_name como categoría por alta repetición de valores
    df_optimized['med_base_name'] = df_optimized['med_base_name'].astype('category')
    
    return df_optimized

### Ejecución del pipeline de `pyxis`

Se encadenan las tres transformaciones mediante `.pipe()` y se imprimen las dimensiones y una muestra del resultado para validación visual.

In [22]:
pyxis_cleaned = (
    df_pyxis
    .pipe(clean_pyxis_names)
    .pipe(clean_and_impute_gsn)
    .pipe(optimize_pyxis_dataset)
)

# Verificación de resultados
print(pyxis_cleaned.info())
print(pyxis_cleaned[['med_base_name', 'gsn']].head(15))

<class 'pandas.DataFrame'>
RangeIndex: 1586053 entries, 0 to 1586052
Data columns (total 8 columns):
 #   Column         Non-Null Count    Dtype         
---  ------         --------------    -----         
 0   subject_id     1586053 non-null  int32         
 1   stay_id        1586053 non-null  int32         
 2   charttime      1586053 non-null  datetime64[us]
 3   med_rn         1586053 non-null  int16         
 4   name           1586053 non-null  str           
 5   gsn_rn         1586053 non-null  int16         
 6   gsn            1586053 non-null  Int64         
 7   med_base_name  1586053 non-null  category      
dtypes: Int64(1), category(1), datetime64[us](1), int16(2), int32(2), str(1)
memory usage: 88.9 MB
None
         med_base_name    gsn
0            ALBUTEROL   5037
1            ALBUTEROL  28090
2             MORPHINE   4080
3             DONNATOL   4773
4   ALUMINUM-MAGNESIUM   2701
5   ALUMINUM-MAGNESIUM   2716
6          ONDANSETRON  15869
7          ONDANSETRON  6

### Exportación del dataset limpio de `pyxis`

Se persiste el resultado en `clean_csv/pyxis_cleaned.csv` para su uso en etapas de feature engineering y construcción de etiquetas.

In [23]:
directory = "clean_csv"
if not os.path.exists(directory):
    os.makedirs(directory)

file_path = os.path.join(directory, "pyxis_cleaned.csv")
pyxis_cleaned.to_csv(file_path, index=False)

print(f"Archivo guardado en: {file_path}")

Archivo guardado en: clean_csv\pyxis_cleaned.csv


## Tabla `triage`

Contiene las constantes vitales y el motivo de consulta registrados en el momento de triaje (primera evaluación en urgencias). Es la principal fuente de features del modelo al corresponder a los primeros 5 minutos.

In [24]:
df_triage = pd.read_csv(DATA / 'triage.csv')
reporte = sv.analyze(df_triage)
reporte.show_html(str(DATA.parent / 'analisis_mimic_ti.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_ti.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Pipeline de limpieza — `triage`

Se normalizan los campos de texto libre, se asignan tipos correctos a identificadores y vitales, y se eliminan duplicados exactos.

In [25]:
def clean_triage_strings(df):
    """Normaliza chiefcomplaint a minúsculas e imputa valores nulos con 'unknown'. pain se excluye: se parsea a numérico en optimize_triage_types."""
    print("--- Paso 1: Limpiando textos libres ---")
    text_cols = ['chiefcomplaint']
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()
            df[col] = df[col].replace('nan', 'unknown')
    return df

In [26]:
def optimize_triage_types(df):
    """Asigna los tipos correctos a identificadores, categorías y variables numéricas. Incluye parsing de pain a escala 0-10."""
    print("--- Paso 2: Optimizando IDs, Categorías y Numéricos ---")

    if 'subject_id' in df.columns:
        df['subject_id'] = df['subject_id'].astype(int)
    if 'stay_id' in df.columns:
        df['stay_id'] = df['stay_id'].astype(int)

    if 'acuity' in df.columns:
        df['acuity'] = df['acuity'].astype('Int64')

    vital_cols = ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp']
    for col in vital_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # pain: texto libre con formatos mixtos ("7", "7/10", "moderate") → extracción del primer número
    if 'pain' in df.columns:
        df['pain'] = (
            df['pain'].astype(str)
            .str.extract(r'(\d+(?:\.\d+)?)', expand=False)
            .astype(float)
        )
        # Valores fuera del rango 0-10 son fisiológicamente imposibles; se tratan como ausentes
        df.loc[df['pain'] > 10, 'pain'] = np.nan

    return df

In [27]:
def run_triage_pipeline(df_input):
    """Función principal del pipeline para la tabla Triage."""
    df = df_input.copy()
    df.columns = df.columns.str.strip()
    
    df = (
        df.pipe(clean_triage_strings)
          .pipe(optimize_triage_types)
    )
    
    # Eliminación de posibles duplicados exactos
    df = df.drop_duplicates()
    
    print(f"Pipeline de Triage completado. Filas finales: {len(df)}")
    return df

In [28]:
df_triage_limpio = run_triage_pipeline(df_triage)
# Guardado
output_dir = "clean_csv"
df_triage_limpio.to_csv(os.path.join(output_dir, "triage_cleaned.csv"), index=False)

--- Paso 1: Limpiando textos libres ---
--- Paso 2: Optimizando IDs, Categorías y Numéricos ---
Pipeline de Triage completado. Filas finales: 425087


## Tabla `vitalsign`

Series temporales de constantes vitales registradas durante la estancia en urgencias. Contiene múltiples filas por `stay_id`. Requiere tratamiento de outliers y, en esta fase exploratoria, imputación por mediana para el análisis univariante.

In [29]:
df_vitalsign = pd.read_csv(DATA / 'vitalsign.csv')
reporte = sv.analyze(df_vitalsign)
reporte.show_html(str(DATA.parent / 'analisis_mimic_vi.html'))

                                             |          | [  0%]   00:00 -> (? left)

Report C:\Users\cuent\Documents\UAX\TFM\TFM\CodigoGit\analisis_mimic_vi.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


### Funciones de detección de outliers

Se implementan dos métodos complementarios: IQR (robusto para distribuciones asimétricas de variables hemodinámicas) y MAD (Z-score modificado, más apropiado para temperatura por su distribución más simétrica).

In [30]:
def detect_outliers_iqr(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    return (series < lower_bound) | (series > upper_bound)

In [31]:
def detect_outliers_mad(series, threshold=3.5):
    median = series.median()
    mad = np.median(np.abs(series - median))
    if mad == 0:
        return pd.Series(False, index=series.index)
    modified_z_scores = 0.6745 * (series - median) / mad
    return np.abs(modified_z_scores) > threshold

### Pipeline de limpieza — `vitalsign`

Encadena la limpieza estructural (drop de `rhythm`, conversión de `charttime`), el tratamiento de ceros y outliers, la optimización de tipos y la eliminación de duplicados.

In [32]:
def clean_structure_and_time(df):
    """Elimina la columna 'rhythm' (sin uso) y convierte charttime a datetime."""
    print("--- Paso 1: Limpieza estructural ---")
    df = df.drop(columns=['rhythm'], errors='ignore')
    
    if 'charttime' in df.columns:
        df['charttime'] = pd.to_datetime(df['charttime'])
    return df

In [33]:
def handle_zeros_and_outliers(df):
    """Sustituye ceros fisiológicamente imposibles por NaN y elimina outliers estadísticos."""
    print("--- Paso 2: Tratando ceros y outliers ---")
    
    cols_vital = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'temperature']
    
    # Ceros en constantes vitales son fisiológicamente imposibles; se tratan como ausentes
    for col in cols_vital:
        if col in df.columns:
            df.loc[df[col] == 0, col] = np.nan
            
    # IQR con multiplicador 1.5 para variables hemodinámicas
    iqr_cols = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp']
    for col in iqr_cols:
        if col in df.columns:
            outliers = detect_outliers_iqr(df[col].dropna(), 1.5)
            df.loc[outliers[outliers].index, col] = np.nan
            
    # MAD para temperatura: distribución más centrada, el Z-score modificado es más robusto
    if 'temperature' in df.columns:
        outliers = detect_outliers_mad(df['temperature'].dropna(), 3.5)
        df.loc[outliers[outliers].index, 'temperature'] = np.nan
        
    return df

In [34]:
def impute_and_flag_missing(df):
    """Imputa por mediana y crea indicadores binarios de imputación por columna."""
    print("--- Paso 3: Imputación por mediana y creación de flags ---")
    cols_to_impute = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'temperature']
    
    for col in cols_to_impute:
        if col in df.columns:
            # Flag binario: 1 si el valor era nulo antes de la imputación
            flag_col = f'imputed_{col}'
            df[flag_col] = df[col].isnull().astype('Int8')
            
            # Imputación con la mediana de la columna
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            
    return df

In [35]:
def optimize_vitalsign_types(df):
    """Optimiza los tipos de datos finales del dataset vitalsign."""
    print("--- Paso 3: Optimizando tipos de datos ---")
    if 'subject_id' in df.columns:
        df['subject_id'] = df['subject_id'].astype(int)
    if 'stay_id' in df.columns:
        df['stay_id'] = df['stay_id'].astype(int)
        
    # 'pain' contiene texto libre; se normaliza antes de convertir a cadena
    if 'pain' in df.columns:
        df['pain'] = df['pain'].fillna('unknown').astype(str).str.strip().str.lower()
        df['pain'] = df['pain'].replace(['nan', ''], 'unknown')
        
    return df

In [36]:
def run_vitalsign_pipeline(df_input):
    """Función orquestadora del pipeline de limpieza de vitalsign."""
    df = df_input.copy()
    df.columns = df.columns.str.strip()
    
    df = (
        df.pipe(clean_structure_and_time)
          .pipe(handle_zeros_and_outliers)
          .pipe(optimize_vitalsign_types)
    )
    
    df = df.drop_duplicates()
    print(f"Pipeline Vitalsign finalizado. Filas: {len(df)}")
    return df

In [37]:
df_vitalsign_limpio = run_vitalsign_pipeline(df_vitalsign)

# Guardar resultado correctamente
output_dir = "clean_csv"
os.makedirs(output_dir, exist_ok=True)
df_vitalsign_limpio.to_csv(os.path.join(output_dir, "vitalsign_cleaned.csv"), index=False)

--- Paso 1: Limpieza estructural ---
--- Paso 2: Tratando ceros y outliers ---
--- Paso 3: Optimizando tipos de datos ---
Pipeline Vitalsign finalizado. Filas: 1564610


### Detección y eliminación de valores atípicos en constantes vitales

Se aplica el método IQR (multiplicador 1.5) para variables hemodinámicas y MAD para temperatura. Los outliers se reemplazan por NaN para su posterior imputación. No se eliminan filas.

In [38]:
def replace_zeros_with_nan(df, cols):
    """Reemplaza ceros por NaN en columnas de constantes vitales."""
    df_clean = df.copy()
    for col in cols:
        df_clean.loc[df_clean[col] == 0, col] = np.nan
    return df_clean
 
cols_vital = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'temperature']
vitalsign = replace_zeros_with_nan(df_vitalsign, cols_vital)
 
print("=" * 70)
print("PASO 2: REEMPLAZAR CEROS POR NaN")
print("=" * 70)
print(f"Missing después de reemplazar ceros:\n{vitalsign.isnull().sum()}\n")

PASO 2: REEMPLAZAR CEROS POR NaN
Missing después de reemplazar ceros:
subject_id           0
stay_id              0
charttime            0
temperature     564990
heartrate        69727
resprate         89456
o2sat           135904
sbp              81259
dbp              81275
rhythm         1504960
pain            443266
dtype: int64



In [39]:
def detect_outliers_iqr(series, multiplier=1.5):
    """
    Detecta outliers mediante el método IQR.
    
    Args:
        series: Serie numérica a evaluar.
        multiplier: Factor multiplicador del IQR. El valor estándar es 1.5.
    
    Returns:
        Máscara booleana, límite inferior, límite superior.
    """
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    
    outliers = (series < lower_bound) | (series > upper_bound)
    return outliers, lower_bound, upper_bound
 
 
def detect_outliers_modified_zscore(series, threshold=3.5):
    """
    Detecta outliers mediante Modified Z-Score basado en MAD.
    
    Más robusto que el Z-score clásico para distribuciones asimétricas.
    
    Args:
        series: Serie numérica a evaluar.
        threshold: Umbral del Z-score modificado. Por defecto 3.5.
    
    Returns:
        Máscara booleana, mediana, MAD.
    """
    median = series.median()
    mad = np.median(np.abs(series - median))
    
    if mad == 0:
        return pd.Series(False, index=series.index), None, None
    
    modified_z_scores = 0.6745 * (series - median) / mad
    outliers = np.abs(modified_z_scores) > threshold
    
    return outliers, median, mad
 
 
def remove_outliers_by_column(df, cols_to_check, method='iqr', multiplier=1.5, mad_threshold=3.5):
    """
    Sustituye outliers por NaN en las columnas indicadas.
    
    Args:
        df: DataFrame de entrada.
        cols_to_check: Lista de columnas a procesar.
        method: 'iqr' o 'mad'.
        multiplier: Multiplicador IQR (solo para method='iqr').
        mad_threshold: Umbral MAD (solo para method='mad').
    
    Returns:
        DataFrame con outliers reemplazados por NaN y diccionario de reporte.
    """
    df_clean = df.copy()
    outlier_report = {}
    
    for col in cols_to_check:
        if col not in df.columns:
            continue
            
        series_clean = df_clean[col].dropna()
        if len(series_clean) < 10:
            continue
        
        if method == 'iqr':
            outliers, lower, upper = detect_outliers_iqr(series_clean, multiplier)
            outlier_indices = series_clean[outliers].index
            outlier_report[col] = {
                'method': 'IQR',
                'lower_bound': lower,
                'upper_bound': upper,
                'n_outliers': len(outlier_indices),
                'pct_outliers': (len(outlier_indices) / len(series_clean)) * 100
            }
        
        elif method == 'mad':
            outliers, median, mad = detect_outliers_modified_zscore(series_clean, mad_threshold)
            outlier_indices = series_clean[outliers].index
            outlier_report[col] = {
                'method': 'Modified Z-Score (MAD)',
                'median': median,
                'mad': mad,
                'n_outliers': len(outlier_indices),
                'pct_outliers': (len(outlier_indices) / len(series_clean)) * 100
            }
        
        df_clean.loc[outlier_indices, col] = np.nan
    
    return df_clean, outlier_report

In [40]:
print("=" * 70)
print("PASO 3: DETECCIÓN Y ELIMINACIÓN DE OUTLIERS")
print("=" * 70)
 
# Frecuencia cardiaca: rango fisiológico normal en urgencias 40-150 bpm
vitalsign_clean, report_hr = remove_outliers_by_column(
    vitalsign, 
    ['heartrate'], 
    method='iqr', 
    multiplier=1.5
)
print("\nHEARTRATE (latidos por minuto)")
print(f"  Rango válido clínico: 40-150 bpm")
print(f"  Outliers detectados: {report_hr['heartrate']['n_outliers']} ({report_hr['heartrate']['pct_outliers']:.2f}%)")
print(f"  Límites IQR: [{report_hr['heartrate']['lower_bound']:.1f}, {report_hr['heartrate']['upper_bound']:.1f}]")
 
# Frecuencia respiratoria: rango normal 8-40 resp/min
vitalsign_clean, report_rr = remove_outliers_by_column(
    vitalsign_clean, 
    ['resprate'], 
    method='iqr', 
    multiplier=1.5
)
print("\nRESPRATE (respiraciones por minuto)")
print(f"  Rango válido clínico: 8-40 rpm")
print(f"  Outliers detectados: {report_rr['resprate']['n_outliers']} ({report_rr['resprate']['pct_outliers']:.2f}%)")
print(f"  Límites IQR: [{report_rr['resprate']['lower_bound']:.1f}, {report_rr['resprate']['upper_bound']:.1f}]")
 
# Saturación de O2: rango válido 85-100%
vitalsign_clean, report_o2 = remove_outliers_by_column(
    vitalsign_clean, 
    ['o2sat'], 
    method='iqr', 
    multiplier=1.5
)
print("\nO2SAT (saturación de oxígeno %)")
print(f"  Rango válido clínico: 85-100%")
print(f"  Outliers detectados: {report_o2['o2sat']['n_outliers']} ({report_o2['o2sat']['pct_outliers']:.2f}%)")
print(f"  Límites IQR: [{report_o2['o2sat']['lower_bound']:.1f}, {report_o2['o2sat']['upper_bound']:.1f}]")
 
# Presión sistólica: rango típico en urgencias 80-180 mmHg
vitalsign_clean, report_sbp = remove_outliers_by_column(
    vitalsign_clean, 
    ['sbp'], 
    method='iqr', 
    multiplier=1.5
)
print("\nSBP (presión sistólica mmHg)")
print(f"  Rango válido clínico: 80-180 mmHg")
print(f"  Outliers detectados: {report_sbp['sbp']['n_outliers']} ({report_sbp['sbp']['pct_outliers']:.2f}%)")
print(f"  Límites IQR: [{report_sbp['sbp']['lower_bound']:.1f}, {report_sbp['sbp']['upper_bound']:.1f}]")
 
# Presión diastólica: rango típico 40-120 mmHg
vitalsign_clean, report_dbp = remove_outliers_by_column(
    vitalsign_clean, 
    ['dbp'], 
    method='iqr', 
    multiplier=1.5
)
print("\nDBP (presión diastólica mmHg)")
print(f"  Rango válido clínico: 40-120 mmHg")
print(f"  Outliers detectados: {report_dbp['dbp']['n_outliers']} ({report_dbp['dbp']['pct_outliers']:.2f}%)")
print(f"  Límites IQR: [{report_dbp['dbp']['lower_bound']:.1f}, {report_dbp['dbp']['upper_bound']:.1f}]")
 
# Temperatura: rango válido 35-42 °C; se utiliza MAD por distribución más simétrica
vitalsign_clean, report_temp = remove_outliers_by_column(
    vitalsign_clean, 
    ['temperature'], 
    method='mad', 
    mad_threshold=3.5
)
print("\nTEMPERATURE (grados Celsius)")
print(f"  Rango válido clínico: 35-42°C")
print(f"  Outliers detectados: {report_temp['temperature']['n_outliers']} ({report_temp['temperature']['pct_outliers']:.2f}%)")

vitalsign_clean = vitalsign_clean.drop(columns=['rhythm'])
 
print(f"\nTotal filas eliminadas por outliers: {len(vitalsign) - len(vitalsign_clean)}")
print(f"Porcentaje de datos eliminados: {((len(vitalsign) - len(vitalsign_clean)) / len(vitalsign) * 100):.2f}%")

PASO 3: DETECCIÓN Y ELIMINACIÓN DE OUTLIERS

HEARTRATE (latidos por minuto)
  Rango válido clínico: 40-150 bpm
  Outliers detectados: 32210 (2.15%)
  Límites IQR: [36.0, 124.0]

RESPRATE (respiraciones por minuto)
  Rango válido clínico: 8-40 rpm
  Outliers detectados: 160095 (10.85%)
  Límites IQR: [13.0, 21.0]

O2SAT (saturación de oxígeno %)
  Rango válido clínico: 85-100%
  Outliers detectados: 26082 (1.83%)
  Límites IQR: [92.5, 104.5]

SBP (presión sistólica mmHg)
  Rango válido clínico: 80-180 mmHg
  Outliers detectados: 28444 (1.92%)
  Límites IQR: [69.5, 185.5]

DBP (presión diastólica mmHg)
  Rango válido clínico: 40-120 mmHg
  Outliers detectados: 20652 (1.39%)
  Límites IQR: [34.5, 110.5]

TEMPERATURE (grados Celsius)
  Rango válido clínico: 35-42°C
  Outliers detectados: 62134 (6.22%)

Total filas eliminadas por outliers: 0
Porcentaje de datos eliminados: 0.00%


In [41]:
print("\n" + "=" * 70)
print("PASO 5: IMPUTACIÓN CON MEDIANA")
print("=" * 70)

vitalsign_imputed = vitalsign_clean.copy()

cols_vital_to_impute = ['heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'temperature']

print("Valores nulos ANTES de la imputación:")
print(vitalsign_imputed[cols_vital_to_impute].isnull().sum())
print("-" * 40)

# Imputación por mediana columna a columna
for col in cols_vital_to_impute:
    missing_count = vitalsign_imputed[col].isnull().sum()
    
    if missing_count > 0:
        fill_value = vitalsign_imputed[col].median()
        vitalsign_imputed[col] = vitalsign_imputed[col].fillna(fill_value)
        
        print(f"{col}: {missing_count} valores imputados con mediana {fill_value:.2f}")

print("-" * 40)
print(f"Missing values totales restantes: {vitalsign_imputed[cols_vital_to_impute].isnull().sum().sum()}")


PASO 5: IMPUTACIÓN CON MEDIANA
Valores nulos ANTES de la imputación:
heartrate      101937
resprate       249551
o2sat          161986
sbp            109703
dbp            101927
temperature    627124
dtype: int64
----------------------------------------
heartrate: 101937 valores imputados con mediana 79.00
resprate: 249551 valores imputados con mediana 17.00
o2sat: 161986 valores imputados con mediana 98.00
sbp: 109703 valores imputados con mediana 126.00
dbp: 101927 valores imputados con mediana 72.00
temperature: 627124 valores imputados con mediana 98.10
----------------------------------------
Missing values totales restantes: 0


In [42]:
print("\n" + "=" * 70)
print("PASO 6: CREAR INDICADORES DE IMPUTACIÓN")
print("=" * 70)
 
# Indicadores binarios: 1 si el valor fue imputado (estaba ausente en vitalsign_clean)
for col in cols_vital_to_impute:
    flag_col_name = f'imputed_{col}'
    vitalsign_imputed[flag_col_name] = vitalsign_clean[col].isnull().astype(int)
 
print("\nIndicadores de imputación creados:")
for col in cols_vital_to_impute:
    flag_col = f'imputed_{col}'
    n_imputed = vitalsign_imputed[flag_col].sum()
    pct_imputed = (n_imputed / len(vitalsign_imputed)) * 100
    print(f"  {flag_col}: {n_imputed} valores imputados ({pct_imputed:.2f}%)")


PASO 6: CREAR INDICADORES DE IMPUTACIÓN

Indicadores de imputación creados:
  imputed_heartrate: 101937 valores imputados (6.52%)
  imputed_resprate: 249551 valores imputados (15.95%)
  imputed_o2sat: 161986 valores imputados (10.35%)
  imputed_sbp: 109703 valores imputados (7.01%)
  imputed_dbp: 101927 valores imputados (6.51%)
  imputed_temperature: 627124 valores imputados (40.08%)


In [43]:
print("\n" + "=" * 70)
print("PASO 7: VALIDACIÓN DE VALORES IMPUTADOS")
print("=" * 70)
 
print("\nEstadísticas de variables vitales DESPUÉS de limpieza e imputación:\n")
print(vitalsign_imputed[cols_vital_to_impute].describe().round(2))
 
# Verificación de rangos fisiológicos razonables
print("\n\nVERIFICACIÓN DE RANGOS FISIOLÓGICOS:")
checks = {
    'heartrate': (40, 150, "latidos/min"),
    'resprate': (8, 40, "resp/min"),
    'o2sat': (85, 100, "%"),
    'sbp': (80, 180, "mmHg"),
    'dbp': (40, 120, "mmHg"),
    'temperature': (35, 42, "°C")
}
 
for col, (min_val, max_val, unit) in checks.items():
    out_of_range = ((vitalsign_imputed[col] < min_val) | (vitalsign_imputed[col] > max_val)).sum()
    if out_of_range > 0:
        print(f"  AVISO: {col}: {out_of_range} valores fuera del rango [{min_val}, {max_val}] {unit}")
    else:
        print(f"  OK: {col}: todos los valores dentro del rango [{min_val}, {max_val}] {unit}")


PASO 7: VALIDACIÓN DE VALORES IMPUTADOS

Estadísticas de variables vitales DESPUÉS de limpieza e imputación:

        heartrate    resprate       o2sat         sbp         dbp  temperature
count  1564610.00  1564610.00  1564610.00  1564610.00  1564610.00   1564610.00
mean        80.06       17.09       98.07      127.42       72.51        98.11
std         15.35        1.50        1.76       19.84       13.60         0.41
min         36.00       13.00       93.00       70.00       35.00        96.58
25%         69.00       16.00       97.00      114.00       63.00        98.00
50%         79.00       17.00       98.00      126.00       72.00        98.10
75%         89.00       18.00      100.00      140.00       81.00        98.20
max        124.00       21.00      104.00      185.00      110.00        99.65


VERIFICACIÓN DE RANGOS FISIOLÓGICOS:
  AVISO: heartrate: 1415 valores fuera del rango [40, 150] latidos/min
  OK: resprate: todos los valores dentro del rango [8, 40] resp/min


La columna `pain` presenta formatos mixtos en MIMIC-IV-ED: valores puramente numéricos ("7"), escalas con denominador ("7/10") y cadenas de texto libre ("moderate", "unknown"). Se implementa un parser basado en expresión regular que extrae el primer número encontrado en la cadena. Los registros no parseables (texto libre sin dígito) devuelven NaN, que se imputa con la mediana de entrenamiento en cada modelo. Los valores superiores a 10 (imposibles en la escala EVA estándar) se convierten igualmente a NaN. El resultado es una variable numérica continua en [0, 10] con el mismo tratamiento de missings que el resto de constantes vitales.

In [44]:
directory = "clean_csv"
if not os.path.exists(directory):
    os.makedirs(directory)

file_path = os.path.join(directory, "vitalsign_cleaned.csv")
vitalsign_imputed.to_csv(file_path, index=False)   # era pyxis_cleaned por error de copypaste

print(f"Archivo guardado en: {file_path}")

Archivo guardado en: clean_csv\vitalsign_cleaned.csv


# Fusión de los datasets de MIMIC-IV-ED

Se integran las seis tablas en un único dataset maestro con granularidad `stay_id`. Las tablas 1:N se agregan antes del merge para mantener la fila única por episodio requerida por los modelos tabulares.

In [45]:
def agrupar_vitalsign(df):
    """Calcula estadísticas resumen (min, max, mean) de las constantes vitales por estancia."""
    print("--- Agrupando Vitalsign ---")
    agg_dict = {
        'heartrate': ['min', 'max', 'mean'],
        'resprate': ['min', 'max', 'mean'],
        'o2sat': ['min', 'mean'],
        'sbp': ['min', 'max', 'mean'],
        'dbp': ['min', 'max', 'mean'],
        'temperature': ['min', 'max', 'mean']
    }
    
    # Solo se agrupan columnas presentes en el dataset limpio
    agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}
    
    vitals_agg = df.groupby('stay_id').agg(agg_dict).reset_index()
    # Aplanado de nombres multiíndice: ('heartrate', 'min') -> 'heartrate_min'
    vitals_agg.columns = ['_'.join(col).strip('_') for col in vitals_agg.columns.values]
    return vitals_agg

In [46]:
def agrupar_medrecon(df):
    """Resume los medicamentos previos a la llegada."""
    print("--- Agrupando Medrecon ---")
    meds_agg = df.groupby('stay_id').agg(
        medrecon_count=('name', 'count'),
        medrecon_categorias=('etcdescription', lambda x: list(set(x.dropna())))
    ).reset_index()
    return meds_agg

In [47]:
def agrupar_pyxis(df):
    """Resume los medicamentos suministrados en urgencias."""
    print("--- Agrupando Pyxis ---")
    pyxis_agg = df.groupby('stay_id').agg(
        pyxis_dispensations_count=('name', 'count'),
        pyxis_unique_meds=('name', lambda x: list(set(x.dropna())))
    ).reset_index()
    return pyxis_agg

In [48]:
def agrupar_diagnosis(df):
    """Extrae el diagnóstico principal y cuenta el total de códigos ICD."""
    print("--- Agrupando Diagnosis ---")
    
    # Diagnóstico principal: registro con seq_num == 1
    df_primary = df[df['seq_num'] == 1][['stay_id', 'icd_code', 'icd_title']].copy()
    df_primary.rename(columns={'icd_code': 'primary_icd_code', 'icd_title': 'primary_icd_title'}, inplace=True)
    
    # Conteo total de diagnósticos y lista de títulos por estancia
    df_counts = df.groupby('stay_id').agg(
        total_diagnoses=('icd_code', 'count'),
        all_icd_titles=('icd_title', lambda x: list(set(x.dropna())))
    ).reset_index()
    
    # Combinación del diagnóstico principal con los conteos agregados
    diag_agg = pd.merge(df_counts, df_primary, on='stay_id', how='left')
    return diag_agg

In [49]:
def fusionar_mimic_ed(df_edstays, df_triage, df_vitals, df_medrecon, df_pyxis, df_diagnosis):
    """Orquesta la unión de las 6 tablas manteniendo edstays como tabla base."""
    print("\nINICIANDO FUSIÓN GLOBAL DE MIMIC-IV-ED...")
    
    # Agregación de tablas 1:N antes del merge
    vitals_agg = agrupar_vitalsign(df_vitals)
    medrecon_agg = agrupar_medrecon(df_medrecon)
    pyxis_agg = agrupar_pyxis(df_pyxis)
    diagnosis_agg = agrupar_diagnosis(df_diagnosis)
    
    df_master = df_edstays.copy()
    
    # Triage es 1:1 con edstays; se deduplica por precaución
    df_triage_unique = df_triage.drop_duplicates(subset=['stay_id'])
    df_master = pd.merge(df_master, df_triage_unique, on=['subject_id', 'stay_id'], how='left', suffixes=('', '_triage'))
    
    # Merge secuencial de las tablas agregadas (1:N)
    tablas_a_unir = [vitals_agg, medrecon_agg, pyxis_agg, diagnosis_agg]
    nombres_tablas = ['Vitalsign', 'Medrecon', 'Pyxis', 'Diagnosis']
    
    for tabla, nombre in zip(tablas_a_unir, nombres_tablas):
        df_master = pd.merge(df_master, tabla, on='stay_id', how='left')
        print(f"  {nombre} fusionada.")
        
    # Estancias sin registros en tablas opcionales reciben conteo 0
    cols_to_fill_zero = ['medrecon_count', 'pyxis_dispensations_count', 'total_diagnoses']
    for col in cols_to_fill_zero:
        if col in df_master.columns:
            df_master[col] = df_master[col].fillna(0).astype(int)
            
    print(f"\nFUSION COMPLETADA. Filas finales: {len(df_master)}")
    return df_master

In [50]:
df_final_tfm = fusionar_mimic_ed(
    df_edstays_limpio, 
    df_triage_limpio, 
    vitalsign_clean, 
    df_medrecon_limpio, 
    pyxis_cleaned, 
    diagnosis_cleaned
 )

# Serialización del dataset maestro en formato Pickle para preservar tipos
df_final_tfm.to_pickle('mimic_ed_maestro.pkl')
print("Archivo guardado como Pickle.")


INICIANDO FUSIÓN GLOBAL DE MIMIC-IV-ED...
--- Agrupando Vitalsign ---
--- Agrupando Medrecon ---
--- Agrupando Pyxis ---
--- Agrupando Diagnosis ---
  Vitalsign fusionada.
  Medrecon fusionada.
  Pyxis fusionada.
  Diagnosis fusionada.

FUSION COMPLETADA. Filas finales: 425087
Archivo guardado como Pickle.


In [51]:
print(f"Dataset cargado. Filas: {df_final_tfm.shape[0]}, Columnas: {df_final_tfm.shape[1]}")
display(df_final_tfm.head(20))

Dataset cargado. Filas: 425087, Columnas: 43


,subject_id,hadm_id,stay_id,intime,outtime,gender,race,arrival_transport,disposition,temperature,...,temperature_max,temperature_mean,medrecon_count,medrecon_categorias,pyxis_dispensations_count,pyxis_unique_meds,total_diagnoses,all_icd_titles,primary_icd_code,primary_icd_title
0,10000032,22595853,33258284,2180-05-06 19:17:00,2180-05-06 23:30:00,F,WHITE,AMBULANCE,ADMITTED,98.40,...,97.7,97.700000,10,[asthma/copd therapy - beta 2-adrenergic agent...,0,NaN,4,"[oth sequela chr liv dis, unspecified viral he...",5728,oth sequela chr liv dis
1,10000032,22841357,38112554,2180-06-26 15:54:00,2180-06-26 21:31:00,F,WHITE,AMBULANCE,ADMITTED,98.90,...,97.9,97.900000,13,[asthma/copd therapy - beta 2-adrenergic agent...,3,"[MORPHINE, ONDANSETRON]",4,"[other ascites, unspecified viral hepatitis c ...",78959,other ascites
2,10000032,25742920,35968195,2180-08-05 20:58:00,2180-08-06 01:44:00,F,WHITE,AMBULANCE,ADMITTED,99.40,...,98.5,98.300000,7,"[analgesic opioid agonists, colonic acidifier ...",6,"[MORPHINE, ALUMINUM-MAGNESIUM HYDROX.-SIMET, D...",3,"[cirrhosis of liver nos, asymptomatic hiv infe...",5715,cirrhosis of liver nos
3,10000032,29079034,32952584,2180-07-22 16:24:00,2180-07-23 05:54:00,F,WHITE,AMBULANCE,HOME,97.80,...,98.4,98.300000,14,"[analgesic opioid agonists, asthma/copd therap...",2,[ALBUTEROL INHALER],3,[unspecified viral hepatitis c without hepatic...,4589,hypotension nos
4,10000032,29079034,39399961,2180-07-23 05:54:00,2180-07-23 14:00:00,F,WHITE,AMBULANCE,ADMITTED,98.70,...,99.0,98.550000,14,"[analgesic opioid agonists, asthma/copd therap...",14,"[LACTULOSE, CEFTRIAXONE (MINI BAG PLUS), DEXTR...",2,"[altered mental status, encephalopathy unspeci...",78097,altered mental status
5,10000084,23052089,35203156,2160-11-20 20:36:00,2160-11-21 03:20:00,M,WHITE,WALK IN,ADMITTED,97.50,...,98.0,97.833333,6,[antiparkinson therapy - monoamine oxidase inh...,1,[PRAVASTATIN],2,"[parkinsons disease, weakness]",R531,weakness
6,10000084,29888819,36954971,2160-12-27 18:32:00,2160-12-28 16:07:00,M,WHITE,AMBULANCE,HOME,98.70,...,98.7,98.300000,8,[antiparkinson therapy - monoamine oxidase inh...,1,[LIDOCAINE JELLY 2% (GLYDO)],2,[unspecified dementia without behavioral distu...,R4182,altered mental status unspecified
7,10000108,27250926,36533795,2163-09-27 16:18:00,2163-09-28 09:04:00,M,WHITE,WALK IN,HOME,98.80,...,99.0,98.500000,1,[penicillin antibiotic - natural],3,"[OXYCODONE-ACETAMINOPHEN, AMPICILLIN-SULBACTAM]",1,[cellulitisabscess mouth],5283,cellulitisabscess mouth
8,10000108,-1,32522732,2163-09-16 16:34:00,2163-09-16 18:13:00,M,WHITE,WALK IN,HOME,98.21,...,98.2,98.200000,0,NaN,0,NaN,1,[local suprficial swellng],7822,local suprficial swellng
9,10000108,-1,39513268,2163-09-24 16:14:00,2163-09-24 21:02:00,M,WHITE,WALK IN,HOME,98.80,...,98.4,98.400000,0,NaN,2,[PENICILLIN V POTASSIUM],1,[cellulitisabscess mouth],5283,cellulitisabscess mouth


# Dataset restringido a los primeros 5 minutos de triaje

Para cumplir el requisito TRIPOD-AI de ausencia de fuga de datos, se construye un dataset que incluye exclusivamente la información disponible en el momento del triaje: datos demográficos de `edstays`, constantes vitales de `triage` y medicación previa de `medrecon`. No se incorporan variables de `vitalsign` ni `diagnosis`, que son posteriores al triage.

In [52]:
def agrupar_medrecon_texto(df_meds):
    """Resume los medicamentos previos como cadena de categorías terapéuticas, compatible con CatBoost."""
    print("Agrupando antecedentes de medicación...")
    meds_agg = df_meds.groupby('stay_id').agg(
        medrecon_count=('name', 'count'),
        # Categorías unidas por espacio para uso directo como feature textual en CatBoost
        medrecon_categorias=('etcdescription', lambda x: ' '.join(set(x.dropna().astype(str))))
    ).reset_index()
    return meds_agg

In [53]:
def crear_dataset_triage_estricto(path_clean):
    """Construye el dataset de entrada para el modelo con información disponible en los primeros 5 minutos."""
    print("--- PIPELINE: TRIAGE 5 MINUTOS ---")
    
    # Solo se cargan los datasets con información anterior al triage
    df_edstays = pd.read_csv(f"{path_clean}/edstays_cleaned.csv")
    df_triage = pd.read_csv(f"{path_clean}/triage_cleaned.csv")
    df_medrecon = pd.read_csv(f"{path_clean}/medrecon_cleaned.csv")
    
    medrecon_agg = agrupar_medrecon_texto(df_medrecon)
    
    print("Fusionando datos de entrada...")
    df_triage_unique = df_triage.drop_duplicates(subset=['stay_id'])
    df_model = pd.merge(df_edstays, df_triage_unique, on=['subject_id', 'stay_id'], how='left')
    df_model = pd.merge(df_model, medrecon_agg, on='stay_id', how='left')
    
    df_model['medrecon_count'] = df_model['medrecon_count'].fillna(0).astype(int)
    df_model['medrecon_categorias'] = df_model['medrecon_categorias'].fillna('sin_datos')
    
    # Variable objetivo: ingreso hospitalario, muerte en urgencias o acuity nivel 1
    cond_ingreso = df_model['hadm_id'] != -1
    cond_muerte = df_model['disposition'].astype(str).str.upper() == 'EXPIRED'
    cond_critico = df_model['acuity'] == 1
    
    df_model['target_deterioro'] = (cond_ingreso | cond_muerte | cond_critico).astype(int)
    
    # Eliminación de columnas con información posterior al triage (data leakage)
    cols_leakage = [
        'subject_id', 'hadm_id', 'stay_id', 'outtime', 'disposition', 'intime'
    ]
    df_model = df_model.drop(columns=[c for c in cols_leakage if c in df_model.columns], errors='ignore')
    
    # Normalización de categóricas para CatBoost
    cat_cols = ['gender', 'race', 'arrival_transport']
    for col in cat_cols:
        if col in df_model.columns:
            df_model[col] = df_model[col].fillna('desconocido').astype(str).str.lower()
            
    if 'chiefcomplaint' in df_model.columns:
        df_model['chiefcomplaint'] = df_model['chiefcomplaint'].fillna('sin_datos').astype(str)

    print(f"Dataset listo. Filas: {df_model.shape[0]}, Predictores: {df_model.shape[1] - 1}")
    return df_model

In [54]:
df_modelo_final = crear_dataset_triage_estricto('clean_csv')

--- PIPELINE: TRIAGE 5 MINUTOS ---
Agrupando antecedentes de medicación...
Fusionando datos de entrada...
Dataset listo. Filas: 425087, Predictores: 14


In [55]:
df_modelo_final.to_csv(CLEAN / 'triage_5min_model_ready.csv', index=False)